# LLM Hosting on Kaggle via Cloudflare Tunnel

Stack: **Kaggle Model** (pre-attached, no download) → **vLLM** (OpenAI-compatible API) → **Cloudflare Tunnel** → your frontend

## Before running — two settings in the right panel:
1. **Accelerator** → GPU T4 x2 *(requires phone verification on your Kaggle account)*
2. **Internet** → On

## Attach a model:
1. Right panel → **Add-ons** → **Models**
2. Search for a model (e.g. `gemma 3`, `llama 3.1`, `deepseek r1`, `phi-4`, `qwen2.5`)
3. Pick a variant that fits in ~28GB VRAM (2× T4): **4B–8B** size, `transformers` framework
4. Click **Attach** — it appears at `/kaggle/input/<model-slug>/...` instantly, no download

The vLLM server exposes `/v1/chat/completions` — the same OpenAI API format your chatbot frontend already expects.

In [1]:
import os, glob, subprocess, shutil

# ── Verify GPU ───────────────────────────────────────────────────────────────
if shutil.which("nvidia-smi"):
    r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                       capture_output=True, text=True)
    print("✓ GPUs:\n ", r.stdout.strip())
else:
    print("✗ No GPU — right panel → Accelerator → GPU T4 x2, then restart")

# ── Auto-detect attached model path ─────────────────────────────────────────
# Use .safetensors as the marker — unique to model weights, works for all models
# including Qwen3 (slug: qwen-3) and Qwen2.5 (slug: qwen2.5)
hits = glob.glob("/kaggle/input/**/*.safetensors", recursive=True)

if hits:
    MODEL_PATH = os.path.dirname(hits[0])
    print(f"✓ Model found: {MODEL_PATH}")
    print(f"  Files: {os.listdir(MODEL_PATH)[:5]} ...")
else:
    MODEL_PATH = None
    print("✗ No model found. Contents of /kaggle/input/models/:")
    try:
        for root, dirs, files in os.walk("/kaggle/input/models"):
            level = root.replace("/kaggle/input/models", "").count(os.sep)
            if level < 4:
                print("  " * level + os.path.basename(root) + "/")
    except Exception as e:
        print(" ", e)







# Install vLLM — OpenAI-compatible inference server
# Takes ~2 min on first run
import subprocess
subprocess.run("pip install vllm -q", shell=True, check=True)
print("✓ vLLM installed")




# 

import subprocess, time, os

assert MODEL_PATH, "Run cell-0 first — model not found"

subprocess.run(["pkill", "-f", "vllm.entrypoints"], capture_output=True)
time.sleep(2)

# ── Fix: FlashInfer JIT needs libcuda.so at link time ──────────────────
# Kaggle has no CUDA toolkit dev files, so the stubs dir is absent.
# The real driver library (libcuda.so.1) lives in the OS lib path.
r = subprocess.run(
    "find /usr/lib/x86_64-linux-gnu /usr/lib /lib -maxdepth 2 -name 'libcuda.so*' 2>/dev/null",
    shell=True, capture_output=True, text=True
)
candidates = [l.strip() for l in r.stdout.strip().split('\n') if l.strip()]
print("libcuda.so* found:", candidates or "NONE")

dest = "/usr/local/cuda/lib64/libcuda.so"
os.makedirs("/usr/local/cuda/lib64", exist_ok=True)

if candidates and not os.path.exists(dest):
    os.symlink(candidates[0], dest)
    print(f"✓ libcuda.so → {candidates[0]}")
elif not candidates:
    # No libcuda.so found anywhere — create a minimal stub via gcc.
    # The linker needs a valid ELF .so to satisfy -lcuda; actual CUDA driver
    # symbols are resolved at runtime from the kernel module, not this file.
    subprocess.run(
        "gcc -shared -fPIC -nostartfiles -Wl,--allow-shlib-undefined "
        "-x c /dev/null -o /usr/local/cuda/lib64/libcuda.so",
        shell=True, check=True
    )
    print("✓ Created minimal libcuda.so stub via gcc")

# Always clear stale failed-build artifacts before starting
subprocess.run("rm -rf /root/.cache/flashinfer/", shell=True)
print("✓ Cleared FlashInfer build cache")
# ───────────────────────────────────────────────────────────────────────

env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL_PATH,
        "--port", "8000",
        "--tensor-parallel-size", "2",
        "--gpu-memory-utilization", "0.88",
        "--max-model-len", "4096",
        "--enforce-eager",       # skips torch.compile — saves ~750MB/GPU, needed on T4
        "--trust-remote-code",
        "--enable-auto-tool-choice",
        "--tool-call-parser", "hermes",
        "--override-generation-config", '{"thinking_budget": 0}',
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
)

print("Starting vLLM (60-90s)...")
for line in proc.stdout:
    print(line, end="")
    if "Application startup complete" in line or "Uvicorn running" in line:
        print("\n✓ vLLM ready on :8000")
        break



✓ GPUs:
  Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
✓ Model found: /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1
  Files: ['model.safetensors.index.json', 'config.json', 'merges.txt', 'model-00005-of-00005.safetensors', 'model-00001-of-00005.safetensors'] ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.2/248.2 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.1 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
pylibcudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.2.0 which is incompatible.
libcuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
rmm-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.2.0 which is incompatible.
cuml-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.2.0 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,

✓ vLLM installed
libcuda.so* found: NONE
✓ Created minimal libcuda.so stub via gcc
✓ Cleared FlashInfer build cache
Starting vLLM (60-90s)...
(APIServer pid=170) INFO 05-18 09:24:27 [utils.py:306] 
(APIServer pid=170) INFO 05-18 09:24:27 [utils.py:306]        █     █     █▄   ▄█
(APIServer pid=170) INFO 05-18 09:24:27 [utils.py:306]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.21.0
(APIServer pid=170) INFO 05-18 09:24:27 [utils.py:306]   █▄█▀ █     █     █     █  model   /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1
(APIServer pid=170) INFO 05-18 09:24:27 [utils.py:306]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=170) INFO 05-18 09:24:27 [utils.py:306] 
(APIServer pid=170) INFO 05-18 09:24:27 [utils.py:240] non-default args: {'enable_auto_tool_choice': True, 'tool_call_parser': 'hermes', 'model': '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1', 'trust_remote_code': True, 'max_model_len': 4096, 'enforce_eager': True, 'override_generation_config': {'thinking_budget': 0}, 'tensor_p

In [3]:
# Install cloudflared
import subprocess
subprocess.run(
    "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb "
    "-O /tmp/cloudflared.deb && dpkg -i /tmp/cloudflared.deb",
    shell=True, check=True
)
print("✓ cloudflared installed")




# Start Cloudflare Quick Tunnel on port 8000 (vLLM)
import subprocess, threading, time, re

cf = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

TUNNEL_URL = None

def read_cf():
    global TUNNEL_URL
    for line in cf.stdout:
        print(line, end="")
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m and not TUNNEL_URL:
            TUNNEL_URL = m.group(0)
            print(f"\n✓ Tunnel: {TUNNEL_URL}")

threading.Thread(target=read_cf, daemon=False).start()
time.sleep(12)
print("TUNNEL_URL =", TUNNEL_URL)





# End-to-end verification
import requests

r = requests.get(f"{TUNNEL_URL}/v1/models", timeout=15)
print("✓ Tunnel reachable:", r.status_code)
print("  Model ID:", r.json()["data"][0]["id"])
print()
print("Use in your chatbot frontend:")
print(f"  Base URL : {TUNNEL_URL}/v1")
print(f"  Model    : {r.json()['data'][0]['id']}")
print(f"  API key  : none (or any string)")

(Reading database ... 124630 files and directories currently installed.)
Preparing to unpack /tmp/cloudflared.deb ...
Unpacking cloudflared (2026.5.0) over (2026.5.0) ...
Setting up cloudflared (2026.5.0) ...
Processing triggers for man-db (2.10.2-1) ...
✓ cloudflared installed
2026-05-18T09:29:51Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-18T09:29:51Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-18T09:29:54Z INF +----------------

In [4]:
# End-to-end verification
import requests

r = requests.get(f"{TUNNEL_URL}/v1/models", timeout=15)
print("✓ Tunnel reachable:", r.status_code)
print("  Model ID:", r.json()["data"][0]["id"])
print()
print("Use in your chatbot frontend:")
print(f"  Base URL : {TUNNEL_URL}/v1")
print(f"  Model    : {r.json()['data'][0]['id']}")
print(f"  API key  : none (or any string)")

✓ Tunnel reachable: 200
  Model ID: /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1

Use in your chatbot frontend:
  Base URL : https://rehab-depth-leone-gallery.trycloudflare.com/v1
  Model    : /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1
  API key  : none (or any string)
2026-05-18T09:38:51Z ERR  error="Incoming request ended abruptly: context canceled" connIndex=0 event=1 ingressRule=0 originService=http://localhost:8000
2026-05-18T09:38:51Z ERR Request failed error="Incoming request ended abruptly: context canceled" connIndex=0 dest=https://rehab-depth-leone-gallery.trycloudflare.com/v1/chat/completions event=0 ip=198.41.192.47 type=http


In [ ]:
# # Keep-alive — leave running for the session duration
# import time, requests

# print(f"Keep-alive running. Tunnel: {TUNNEL_URL}")
# while True:
#     try:
#         requests.get("http://localhost:8000/v1/models", timeout=5)
#     except Exception as e:
#         print("vLLM ping failed:", e)
#     time.sleep(30)